# 02 — Chemistry Comparison: Emulator vs FastChem vs ExoGibbs

Forward-only comparison of three chemistry backends on a single random test
profile and on three in-distribution Guillot profiles:

1. **Emulator** — the exported transformer bundle.
2. **Live FastChem** — subprocess rerun via
   `classical_reference.run_fastchem_online`, using the same input-writing
   logic as data generation (this is the oracle the emulator was trained
   against).
3. **ExoGibbs** — `chemsetup_matched_to_fastchem(...)` Gibbs-free-energy
   minimizer pinned to FastChem's `logK_wo_ions.dat` thermo file.

This notebook is gradient-free and JIT-free — it is the cleanest place to
read absolute log10 disagreement between the three backends. The gradient
gates and retrieval comparisons live in notebooks 04–06.


In [ ]:
# ExoJAX runs in float64 by default; the emulator is trained in float32 and
# JAX down-casts at the boundary, which matches training precision.
from jax import config
config.update("jax_enable_x64", True)


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Resolve the project root whether this notebook is launched from
# `exojax_demo/` or from the repo root.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "exojax_demo":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Pick the bundle to load. Override via the VULCAN_DEMO_MODEL env var
# (used by the PBS submission script in supercomputer_cmds/).
import os
MODEL = os.environ.get("VULCAN_DEMO_MODEL", "fastchem")
BUNDLE_PATH = (PROJECT_ROOT / "models" / MODEL / "best_exported.npz").resolve()
assert BUNDLE_PATH.exists(), f"bundle not found at {BUNDLE_PATH}"

# Put the distribution root on sys.path so `src` imports resolve.
DIST_ROOT = BUNDLE_PATH.parents[2]
assert (DIST_ROOT / "src").is_dir(), (
    f"src not found at {DIST_ROOT / 'src'} — the distribution must keep "
    "`src/` next to `models/`."
)
if str(DIST_ROOT) not in sys.path:
    sys.path.insert(0, str(DIST_ROOT))

# Project-shipped matplotlib style (ships in this folder).
_STYLE = Path("science.mplstyle")
if _STYLE.exists():
    plt.style.use(str(_STYLE))

print(f"MODEL       : {MODEL}")
print(f"BUNDLE_PATH : {BUNDLE_PATH}")


## Load the bundle and build the three backend handles

`build_exogibbs_element_vector(..., mode="fastchem_proxy")` is the single
source of truth for translating a 5-element `{He_H, C_H, O_H, N_H, S_H}` dict
into the 28-element vector ExoGibbs expects under the apples-to-apples
contract. See `spec.md`.


In [ ]:
from src.constants import SOLAR_ABUNDANCES
from src.models.standalone_inference import load_model
from src.models.classical_reference import (
    EPSILON,
    build_exogibbs_element_vector,
    build_exogibbs_species_indices,
    chemsetup_matched_to_fastchem,
    mean_abs_log10_error,
    resolve_vulcan_source_root,
    run_fastchem_online,
)
from exogibbs.api.equilibrium import EquilibriumOptions, equilibrium_profile
from exojax.utils.zsol import nsol

# `fastchem_proxy` keeps the ExoGibbs background pinned to the same
# refractory abundances FastChem wrote during training. Use `aas_fixed`
# only for legacy AAG21 comparisons (off-contract).
EXOGIBBS_ELEMENT_MODE = "fastchem_proxy"

model = load_model(BUNDLE_PATH)
species_labels = model.species
FASTCHEM_SOURCE_ROOT = resolve_vulcan_source_root(model.config, project_root=PROJECT_ROOT)

chem = chemsetup_matched_to_fastchem(FASTCHEM_SOURCE_ROOT)
EG_IDX = build_exogibbs_species_indices(chem, species_labels)
EG_OPTS = EquilibriumOptions(epsilon_crit=1e-11, max_iter=1000, method="vmap_cold")
SOLAR_FOR_EG = nsol()

ML_SOLAR = {key: float(value) for key, value in SOLAR_ABUNDANCES.items()}
print(f"ExoGibbs mode  : {EXOGIBBS_ELEMENT_MODE}")
print(f"ExoGibbs species in chem : {len(chem.species)}")
print(f"ExoGibbs elements        : {chem.elements}")


## Helper: run all three backends on a (pressure, temperature, X/H) point


In [ ]:
def run_three_backends(pressure_bar, temperature_k, global_inputs):
    """Return three (nz, n_species) VMR tables for the same input point."""
    vmr_emulator = np.asarray(
        model.predict_fastchem_profile(
            pressure_bar=pressure_bar,
            temperature_k=temperature_k,
            global_inputs=global_inputs,
            return_log10=False,
        )
    )
    vmr_fastchem = run_fastchem_online(
        FASTCHEM_SOURCE_ROOT,
        np.asarray(pressure_bar),
        np.asarray(temperature_k),
        global_inputs,
        species_labels,
        model.config,
    )
    eg_element_vector = build_exogibbs_element_vector(
        chem,
        global_inputs,
        solar_abundances=SOLAR_FOR_EG,
        mode=EXOGIBBS_ELEMENT_MODE,
    )
    vmr_exogibbs = np.asarray(
        equilibrium_profile(
            chem,
            np.asarray(temperature_k),
            np.asarray(pressure_bar),
            eg_element_vector,
            Pref=1.0,
            options=EG_OPTS,
        ).x[:, EG_IDX]
    )
    return vmr_emulator, vmr_fastchem, vmr_exogibbs


## Random test profile: stored target vs all three backends

Pulls one random raw run from the test split and overlays the stored target
together with the three backends. Writes a single-figure 2-panel plot.


In [ ]:
from src.models.classical_reference import (
    load_fastchem_test_context,
    load_fastchem_test_case,
)

ctx = load_fastchem_test_context(
    BUNDLE_PATH, model.config, project_root=PROJECT_ROOT, require_raw=True,
)
eligible = [rid for rid in ctx.split.run_ids if rid in ctx.raw_run_ids]
run_id = str(np.random.default_rng().choice(eligible))
case = load_fastchem_test_case(ctx, run_id)

vmr_emu, vmr_fc, vmr_eg = run_three_backends(
    case.pressure_bar, case.temperature_k, case.raw_globals or case.global_inputs,
)
stored = np.clip(case.stored_target_ymix, EPSILON, None)

phot = int(np.argmin(np.abs(case.pressure_bar - 0.1)))
print(f"run id : {case.run_id}  ({case.pressure_bar.size} levels)")
print()
print(f"mean |Δ log10 VMR|")
print(f"  emulator  vs stored    = {mean_abs_log10_error(vmr_emu, stored):.4f}")
print(f"  emulator  vs FastChem  = {mean_abs_log10_error(vmr_emu, vmr_fc):.4f}")
print(f"  emulator  vs ExoGibbs  = {mean_abs_log10_error(vmr_emu, vmr_eg):.4f}")
print(f"  FastChem  vs ExoGibbs  = {mean_abs_log10_error(vmr_fc, vmr_eg):.4f}")
print()
print(f"VMR @ P = {case.pressure_bar[phot]:.3f} bar:")
print(f"  {'name':>5s}  {'stored':>11s}  {'emulator':>11s}  {'fastchem':>11s}  {'exogibbs':>11s}")
for name in ("H2", "He", "H2O", "CO", "CO2", "CH4", "NH3", "H2S"):
    if name not in species_labels:
        continue
    i = species_labels.index(name)
    print(f"  {name:>5s}  {stored[phot, i]:>11.3e}  {vmr_emu[phot, i]:>11.3e}  "
          f"{vmr_fc[phot, i]:>11.3e}  {vmr_eg[phot, i]:>11.3e}")


In [ ]:
from matplotlib.lines import Line2D

fig, ax_mix = plt.subplots(
    1, 1,
    figsize=(13, 8),
    constrained_layout=True,
)

colors = plt.cm.tab20(np.linspace(0.0, 1.0, len(species_labels)))

method_styles = {
    "ML": {
        "data": vmr_emu,
        "ls": "-",
        "lw": 4.2,
        "alpha": 0.35,
        "zorder": 1,
    },
    "FastChem": {
        "data": vmr_fc,
        "ls": (0, (10, 5)),
        "lw": 3.6,
        "alpha": 0.98,
        "zorder": 3,
    },
    "ExoGibbs": {
        "data": vmr_eg,
        "ls": (0, (1.2, 4.5)),
        "lw": 3.6,
        "alpha": 0.98,
        "zorder": 4,
    },
}

for i, name in enumerate(species_labels):
    color = colors[i]
    for method_name, style in method_styles.items():
        ax_mix.plot(
            np.clip(style["data"][:, i], EPSILON, None),
            case.pressure_bar,
            color=color,
            lw=style["lw"],
            ls=style["ls"],
            alpha=style["alpha"],
            zorder=style["zorder"],
            solid_capstyle="round",
            dash_capstyle="round",
        )

ax_mix.set_xscale("log")
ax_mix.set_xlim(1.0e-10, 2.0)
ax_mix.set_xlabel("Mixing ratio")
ax_mix.set_ylabel("Pressure (bar)")
ax_mix.set_yscale("log")
ax_mix.invert_yaxis()
ax_mix.set_title("Mixing ratios: ML vs FastChem vs ExoGibbs")
ax_mix.grid(False)

ax_mix.tick_params(axis="both", which="major", labelsize=10, width=1.2, length=6)
ax_mix.tick_params(axis="both", which="minor", width=0.8, length=3)
for spine in ax_mix.spines.values():
    spine.set_linewidth(1.1)

# Species legend
species_handles = [
    Line2D([0], [0], color=colors[i], lw=5.0, label=name)
    for i, name in enumerate(species_labels)
]

species_legend = ax_mix.legend(
    handles=species_handles,
    title="Species",
    fontsize=8,
    title_fontsize=9,
    ncol=2,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False,
    handlelength=2.8,
    columnspacing=1.4,
)
ax_mix.add_artist(species_legend)

# Model legend
method_handles = [
    Line2D([0], [0], color="black", lw=4.2, ls="-", alpha=0.35, label="ML"),
    Line2D([0], [0], color="black", lw=3.6, ls=(0, (10, 5)), label="FastChem"),
    Line2D([0], [0], color="black", lw=3.6, ls=(0, (1.2, 4.5)), label="ExoGibbs"),
]

ax_mix.legend(
    handles=method_handles,
    title="Model",
    fontsize=9,
    title_fontsize=10,
    loc="upper right",
    frameon=False,
    handlelength=4.2,
)

fig.suptitle(case.run_id, fontsize=12)
plt.show()

## Multi-profile sweep: three in-distribution Guillot profiles

Same three backends on three Piette+Madhusudhan (2019) Guillot profiles
sampled from inside the training distribution
(`src/data_generation/sampling.py`). Hot / Warm / Cool profiles stress the
PT-shape direction of the surrogate while keeping abundances fixed at solar.


In [ ]:
from src.data_generation.sampling import (
    _apply_upper_atmosphere_modification,
    _guillot_temperature,
)


def guillot_profile(pressure_bar, *, t_int, t_eq, log10_delta, log10_gamma, alpha, log10_p_trans):
    """Build an in-distribution Guillot profile.

    Parameters are drawn from the same analytic sampler that generated the
    training data. Power-law profiles (T = A * p^alpha) sit *outside* the
    training distribution and produce 0.05–1.3 dex emulator drift, so we
    avoid them here.
    """
    T = _guillot_temperature(
        pressure_bar,
        delta=10.0 ** log10_delta,
        gamma=10.0 ** log10_gamma,
        t_int_k=t_int,
        t_eq_k=t_eq,
    )
    return _apply_upper_atmosphere_modification(
        pressure_bar, T, alpha=alpha, p_trans_bar=10.0 ** log10_p_trans,
    )


pressures = np.logspace(2.0, -6.0, 50)
profiles = [
    ("Hot Guillot",
     guillot_profile(pressures, t_int=400.0, t_eq=2500.0,
                     log10_delta=-2.5, log10_gamma=-0.4,
                     alpha=0.3, log10_p_trans=-1.0)),
    ("Warm Guillot",
     guillot_profile(pressures, t_int=250.0, t_eq=1500.0,
                     log10_delta=-2.0, log10_gamma=-0.3,
                     alpha=0.2, log10_p_trans=-2.0)),
    ("Cool Guillot",
     guillot_profile(pressures, t_int=150.0, t_eq=800.0,
                     log10_delta=-1.5, log10_gamma=-0.3,
                     alpha=0.15, log10_p_trans=-2.5)),
]


In [ ]:
SHOW_SPECIES = [s for s in ("H2", "He", "H2O", "CO", "CH4", "NH3") if s in species_labels]

fig, axes = plt.subplots(
    len(profiles), 2,
    figsize=(13, 4 * len(profiles)),
    sharey="row",
    gridspec_kw={"width_ratios": [1, 2.4], "wspace": 0.0, "hspace": 0.35},
)

for row, (label, Tarr) in enumerate(profiles):
    vmr_emu, vmr_fc, vmr_eg = run_three_backends(pressures, Tarr, ML_SOLAR)
    drift_ml_fc = mean_abs_log10_error(vmr_emu, vmr_fc)
    drift_fc_eg = mean_abs_log10_error(vmr_fc, vmr_eg)

    ax_pt, ax_mix = axes[row]
    ax_pt.plot(Tarr, pressures, color="black", lw=1.6)
    ax_pt.set_yscale("log")
    ax_pt.invert_yaxis()
    ax_pt.set_xlabel("Temperature (K)")
    ax_pt.set_ylabel("Pressure (bar)")
    ax_pt.set_title(label, fontsize=10)
    ax_pt.set_xlim(0, 2999)

    palette = plt.cm.tab10(np.linspace(0.0, 0.9, len(SHOW_SPECIES)))
    for color, name in zip(palette, SHOW_SPECIES):
        i = species_labels.index(name)
        ax_mix.plot(np.clip(vmr_emu[:, i], EPSILON, None), pressures,
                    color=color, lw=1.4, label=name)
        ax_mix.plot(np.clip(vmr_fc[:, i], EPSILON, None), pressures,
                    color=color, lw=1.0, ls=":")
        ax_mix.plot(np.clip(vmr_eg[:, i], EPSILON, None), pressures,
                    color=color, lw=1.0, ls="--")
    ax_mix.set_xscale("log")
    ax_mix.set_xlim(1.0e-15, 3.0)
    ax_mix.set_xlabel("Mixing ratio")
    ax_mix.set_title(
        f"emulator (solid) | FastChem (dotted) | ExoGibbs (dashed) "
        f"— |Δlog10| ml-fc={drift_ml_fc:.3f}, fc-eg={drift_fc_eg:.3f}",
        fontsize=10,
    )
    ax_mix.legend(fontsize=8, ncol=3, loc="lower left")

fig.tight_layout()
plt.show()
